In [1]:
import mediapipe as mp
import cv2
import numpy as np
import os
import time
import tensorflow

In [2]:
mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(min_detection_confidence=0.4,min_tracking_confidence=0.4)
drawing = mp.solutions.drawing_utils

In [3]:
def detection(image , model ):
    image = cv2.cvtColor(image,cv2.COLOR_BGR2RGB)
    image.flags.writeable = False
    results = model.process(image)
    image.flags.writeable = True
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    return image,results


In [4]:
def draw(image,results):
    drawing.draw_landmarks(image,results.pose_landmarks,mp_holistic.POSE_CONNECTIONS,drawing.DrawingSpec((129,45,66),thickness=2,circle_radius=1),drawing.DrawingSpec((129,45,66),thickness=2,circle_radius=1))
    drawing.draw_landmarks(image,results.left_hand_landmarks,mp_holistic.HAND_CONNECTIONS)
    drawing.draw_landmarks(image,results.right_hand_landmarks,mp_holistic.HAND_CONNECTIONS)

In [5]:
def extract_landmarks(results):
    POSE_FEATURES = 33 * 2
    HAND_FEATURES = 21 * 2
    TOTAL_FEATURES = POSE_FEATURES + HAND_FEATURES + HAND_FEATURES
    if results.pose_landmarks:
        pose_landmark = results.pose_landmarks.landmark
        refer_x = pose_landmark[0].x
        refer_y = pose_landmark[0].y
    else:
        return np.zeros(TOTAL_FEATURES)
    
    def normalize(landmarks,refer_x,refer_y):
        features = []
        for lm in landmarks:
            norm_x = lm.x - refer_x
            norm_y = lm.y - refer_y
            features.extend([norm_x,norm_y])
        return np.array(features,dtype=np.float32)
    
    pose_featuers = normalize(pose_landmark,refer_x,refer_y)
    
    if results.left_hand_landmarks:
        left_hand_featuers = normalize(results.left_hand_landmarks.landmark,refer_x,refer_y)
    else:
        left_hand_featuers = np.zeros(HAND_FEATURES,dtype=np.float32)

    if results.right_hand_landmarks:
        right_hand_featuers = normalize(results.right_hand_landmarks.landmark,refer_x,refer_y)
    else:
        right_hand_featuers = np.zeros(HAND_FEATURES,dtype=np.float32)
    final = np.concatenate([pose_featuers,left_hand_featuers,right_hand_featuers]).astype(np.float32)
    return final

In [ ]:
def collecter(guestur_name: str, no_videos: int, start_from: int = 0):
    data_path = "training_data"
    no_frames = 30

    for vid in range(start_from, start_from + no_videos):
        os.makedirs(os.path.join(data_path, guestur_name, str(vid)), exist_ok=True)

    cam = cv2.VideoCapture(0, cv2.CAP_DSHOW)

    for video in range(start_from, start_from + no_videos):
        for frame in range(no_frames):
            success, image = cam.read()
            if not success:
                break

            image = cv2.flip(image, 1)
            image, results = detection(image, holistic)
            draw(image, results)

            # HUD
            total = start_from + no_videos - 1
            info_text = f"{guestur_name} | Video {video}/{total}"
            cv2.putText(image, info_text, (15, 12),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 0, 255), 1, cv2.LINE_AA)

            if frame == 0:
                cv2.putText(image, 'GET READY...', (120, 200),
                            cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 4, cv2.LINE_AA)
                cv2.imshow('Collector', image)
                cv2.waitKey(3000)
            else:
                cv2.imshow('Collector', image)

            features = extract_landmarks(results)
            path = os.path.join(data_path, guestur_name, str(video), str(frame))
            np.save(path, features)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                cam.release()
                cv2.destroyAllWindows()
                print(f"⛔ Stopped at video {video}, frame {frame}")
                return

    cam.release()
    cv2.destroyAllWindows()
    print(f"✅ Done: videos {start_from}→{start_from + no_videos - 1} saved for '{guestur_name}'")

In [ ]:
collecter("hello", 40, start_from=0)

collecter("hello", 40, start_from=40)

collecter("hello", 40, start_from=800)

In [12]:
model = tensorflow.keras.models.load_model(r"C:\Users\Cloud\Downloads\MEDIAPIPE\.ipynb_checkpoints\test model 2 0.83.h5")

In [14]:
actions = np.array(["Good","THANKS","THINKING"]) 
frames = []

cam = cv2.VideoCapture(0, cv2.CAP_DSHOW)
display_until = 0
last_pred_time = 0
bond = 0.70
while cam.isOpened():
    success, frame = cam.read()
    if not success:
        break
    
    frame = cv2.flip(frame, 1)
    image, results = detection(frame, holistic)
    draw(image, results)
    landmarks = extract_landmarks(results)
    frames.append(landmarks)
    frames = frames[-30:]

    if len(frames) == 30 and (time.time() - last_pred_time) >= 2:
        prediction = model.predict(np.expand_dims(frames, axis=0), verbose=0) 
        confident = np.max(prediction)
        predicted_action = actions[np.argmax(prediction)]
        display_until = time.time() + 1
        last_pred_time = time.time()

    if time.time() <= display_until and predicted_action:
        if confident>=bond:
            cv2.putText(image, predicted_action, (200, 450), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 200), 2)
        else:
            prediction = " "
    cv2.imshow('holistic', image)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cam.release()
cv2.destroyAllWindows()